# Gradient Boosting (AdaBoost & GradientBoosting)
**Repositori**: Machine Learning
**Topik**: Implementasi AdaBoost dan Gradient Boosting untuk prediksi penyakit jantung
**Dataset**: heart_disease_data.csv
---
**Pendahuluan**: Ensemble learning dengan teknik boosting menggabungkan banyak weak learner (biasanya decision tree) secara sekuensial untuk membentuk strong learner.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Load & EDA


In [ ]:
df = pd.read_csv('../../data/heart_disease_data.csv')
print('Shape:', df.shape)
print(df.head())
print(df.info())
print(df['target'].value_counts())
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korelasi Fitur Heart Disease')
plt.tight_layout()
plt.show()


## 3. Data Preparation


In [ ]:
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')


## 4. AdaBoost


In [ ]:
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)
ada = AdaBoostClassifier(estimator=base_estimator, n_estimators=100, learning_rate=1.0, random_state=42)
ada.fit(X_train_scaled, y_train)
y_pred_ada = ada.predict(X_test_scaled)
print('=== AdaBoost ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_ada):.4f}')
print(classification_report(y_test, y_pred_ada))
cm = confusion_matrix(y_test, y_pred_ada)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - AdaBoost')
plt.show()


## 5. Gradient Boosting


In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_train)
y_pred_gb = gb.predict(X_test_scaled)
print('=== Gradient Boosting ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_gb):.4f}')
print(classification_report(y_test, y_pred_gb))
cm = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Gradient Boosting')
plt.show()


## 6. Perbandingan dengan Cross Validation


In [ ]:
models = {'AdaBoost': ada, 'Gradient Boosting': gb}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f'{name}: {scores.mean():.4f} +/- {scores.std():.4f}')


## 7. Feature Importance


In [ ]:
for name, model in [('AdaBoost', ada), ('Gradient Boosting', gb)]:
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    importances.plot(kind='bar')
    plt.title(f'Feature Importance - {name}')
    plt.tight_layout()
    plt.show()


## 8. ROC Curve


In [ ]:
plt.figure(figsize=(10, 6))
for name, model in [('AdaBoost', ada), ('Gradient Boosting', gb)]:
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Perbandingan Boosting')
plt.legend()
plt.show()


## 9. Kesimpulan
Gradient Boosting umumnya memberikan performa lebih baik dibanding AdaBoost karena pendekatan pseudo-residual-nya. Feature importance membantu mengidentifikasi fitur paling berpengaruh pada prediksi penyakit jantung.
